In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn.utils.prune as prune
import time
import copy

In [11]:
transform = transforms.ToTensor()

train_data = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_data = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)

100%|██████████| 170M/170M [00:01<00:00, 105MB/s]


In [12]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(32*32*3, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

In [13]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(2):
    total_loss = 0
    for x, y in train_loader:
        optimizer.zero_grad()
        output = model(x)
        loss = loss_fn(output, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print("Epoch:", epoch+1, "Loss:", total_loss)

Epoch: 1 Loss: 1463.8534404039383
Epoch: 2 Loss: 1319.0750761032104


In [14]:
correct = 0
total = 0

start = time.time()

with torch.no_grad():
    for x, y in test_loader:
        output = model(x)
        pred = output.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

end = time.time()

print("Original Accuracy:", correct/total)
print("Original Time:", end-start)

Original Accuracy: 0.4146
Original Time: 1.5996601581573486


In [15]:
pruned_model = copy.deepcopy(model)

prune.l1_unstructured(pruned_model[1], name="weight", amount=0.4)

correct = 0
total = 0

with torch.no_grad():
    for x, y in test_loader:
        output = pruned_model(x)
        pred = output.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

print("Pruned Accuracy:", correct/total)

Pruned Accuracy: 0.4153


In [16]:
student = nn.Sequential(
    nn.Flatten(),
    nn.Linear(32*32*3, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

In [17]:
optimizer = optim.Adam(student.parameters(), lr=0.001)

for epoch in range(2):
    for x, y in train_loader:
        optimizer.zero_grad()

        teacher_output = model(x)
        student_output = student(x)

        loss = nn.functional.mse_loss(student_output, teacher_output)

        loss.backward()
        optimizer.step()

In [18]:
correct = 0
total = 0

with torch.no_grad():
    for x, y in test_loader:
        output = student(x)
        pred = output.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

print("Student Accuracy:", correct/total)

Student Accuracy: 0.4068
